In [1]:
import torch
from torch import nn
x=torch.randn(32,10,5)
lstm=nn.LSTM(input_size=5,hidden_size=10,batch_first=True)
output,(hidden,cell)=lstm(x)
print("Input shape:",x.shape)
print("Output shape:",output.shape)
print("Hidden shape:",hidden.shape)
print("Cell shape:",cell.shape)

Input shape: torch.Size([32, 10, 5])
Output shape: torch.Size([32, 10, 10])
Hidden shape: torch.Size([1, 32, 10])
Cell shape: torch.Size([1, 32, 10])


In [14]:
import torch
from torch import nn
from torchvision import transforms,datasets
from torch.utils.data import DataLoader

## Data Pre-Processing

In [19]:
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
train_transform=transforms.Compose([transforms.RandomRotation(degrees=(-30,30)),transforms.ToTensor()])
test_transform=transforms.Compose([transforms.ToTensor()])
train_dataset=datasets.MNIST(train=True,root="data",download=True,transform=train_transform)
test_dataset=datasets.MNIST(train=False,root="data",download=True,transform=test_transform)
train_loader=DataLoader(dataset=train_dataset,batch_size=64,shuffle=True)
test_loader=DataLoader(dataset=test_dataset,batch_size=64,shuffle=False)
print(next(iter(train_loader))[0].shape)

torch.Size([64, 1, 28, 28])


## Model

In [25]:
class LSTM(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm=nn.LSTM(input_size=28,hidden_size=25,num_layers=3,batch_first=True)
        self.fc=nn.Linear(25,10)
    def forward(self,x):
        output,(hidden,cell)=self.lstm(x)
        output=self.fc(output[:,-1,:])
        return output

## Training and Validation

In [27]:
epoch=5
model=LSTM()
optimizer=torch.optim.Adam(model.parameters(),lr=0.001)
loss_fn=nn.CrossEntropyLoss()
model=model.to(device)
for i in range(epoch):
    model.train()
    total_loss=0
    for data,result in train_loader:
        data=data.squeeze(1)
        optimizer.zero_grad()
        data=data.to(device)
        result=result.to(device)
        predicted_result=model(data)
        batch_loss=loss_fn(predicted_result,result)
        batch_loss.backward()
        optimizer.step()
        total_loss+=batch_loss.item()
    model.eval()
    total=0
    correct=0
    for data,result in test_loader:
        data=data.squeeze(1)
        data=data.to(device)
        result=result.to(device)
        predicted_result=model(data)
        correct+=(predicted_result.argmax(1)==result).type(torch.float).sum().item()
        total+=result.size(0)
    print(f"Epoch {i+1}/{epoch} Loss: {total_loss/len(train_loader)} Accuracy: {correct/len(test_dataset)}")

Epoch 1/5 Loss: 1.023718754492843 Accuracy: 0.8703
Epoch 2/5 Loss: 0.3623489820793557 Accuracy: 0.9437
Epoch 3/5 Loss: 0.24424507775540544 Accuracy: 0.956
Epoch 4/5 Loss: 0.19168605814491319 Accuracy: 0.9694
Epoch 5/5 Loss: 0.15679899353716673 Accuracy: 0.9733
